# v64b — KTF³-R Galaxy Spin Alignment Test
**Data:** Galaxy Zoo 2, Hart et al. (2016) — gz2_hart16.csv.gz
**Prediction:** Spin alignment along KTF³ axis (l=277°, b=-31°)
**OSF:** osf.io/42nks | **GitHub:** github.com/Andrew-Cot/KTF3-Analyse

In [ ]:
# === STEP 1: DECOMPRESS (always runs first) ===
import gzip, shutil, os

GZ_FILE  = 'gz2_hart16.csv.gz'
CSV_FILE = 'gz2_hart16.csv'

if not os.path.exists(CSV_FILE) or os.path.getsize(CSV_FILE) < 1000000:
    if os.path.exists(GZ_FILE):
        print(f'Decompressing {GZ_FILE} ({os.path.getsize(GZ_FILE)/1e6:.0f} MB)...')
        with gzip.open(GZ_FILE, 'rb') as fi, open(CSV_FILE, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        print(f'Done! CSV size: {os.path.getsize(CSV_FILE)/1e6:.0f} MB')
    else:
        print('ERROR: gz2_hart16.csv.gz not found in Colab files!')
        print('Please upload gz2_hart16.csv.gz using the Files panel (left sidebar)')
        raise FileNotFoundError('gz2_hart16.csv.gz missing')
else:
    print(f'CSV already exists: {os.path.getsize(CSV_FILE)/1e6:.0f} MB')

In [ ]:
# === STEP 2: LOAD AND INSPECT ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from astropy.coordinates import SkyCoord
import astropy.units as u

print('Loading gz2_hart16.csv...')
gz2 = pd.read_csv('gz2_hart16.csv')
print(f'Loaded: {len(gz2):,} galaxies')
print(f'Columns ({len(gz2.columns)} total):')
for c in gz2.columns:
    print(f'  {c}')

In [ ]:
# === STEP 3: KTF3 AXIS + ANGULAR SEPARATION ===
l_axis, b_axis = 277.0, -31.0
theta_R = 25.7

axis = SkyCoord(l=l_axis*u.deg, b=b_axis*u.deg, frame='galactic')
ra_ax  = axis.icrs.ra.deg
dec_ax = axis.icrs.dec.deg
print(f'KTF3 axis: l={l_axis}, b={b_axis} -> RA={ra_ax:.2f}, Dec={dec_ax:.2f}')

# Angular separation for all galaxies
ra  = gz2['ra'].values
dec = gz2['dec'].values
good = np.isfinite(ra) & np.isfinite(dec)
gz2 = gz2[good].copy().reset_index(drop=True)
ra  = gz2['ra'].values
dec = gz2['dec'].values

ra_r  = np.radians(ra)
dec_r = np.radians(dec)
cos_s = np.sin(dec_r)*np.sin(np.radians(dec_ax)) + np.cos(dec_r)*np.cos(np.radians(dec_ax))*np.cos(ra_r - np.radians(ra_ax))
sep   = np.degrees(np.arccos(np.clip(cos_s, -1, 1)))

NEAR, FAR = 30.0, 60.0
near = sep < NEAR
far  = sep > FAR

print(f'Total galaxies: {len(sep):,}')
print(f'Near axis (<{NEAR}deg): {near.sum():,}')
print(f'Far  axis (>{FAR}deg):  {far.sum():,}')

In [ ]:
# === STEP 4: FIND BEST SPIN COLUMNS ===
# Print all columns with their means to identify spin proxies
print('Searching for chirality/spin columns...')
print()

# t11 in GZ2 Willett2013 = arm winding (not chirality)
# Chirality (CW/CCW) is in GZ1, not Hart16
# Best proxies in Hart16:
# 1. t01_a02 = features/disk (spiral fraction)
# 2. t11 = arm winding asymmetry
# 3. t04 = bar fraction

fraction_cols = [c for c in gz2.columns if 'weighted_fraction' in c or 'fraction' in c]
print(f'Fraction columns ({len(fraction_cols)}):')
for c in fraction_cols:
    v = gz2[c].dropna()
    print(f'  {c}: mean={v.mean():.3f}, nonzero={( v>0.1).sum():,}')

# Identify the key columns:
spiral_col = None
for c in gz2.columns:
    if 't01' in c and 'a02' in c and 'fraction' in c:
        spiral_col = c
        break

t11_cols = [c for c in gz2.columns if 't11' in c and 'fraction' in c]

print(f'\nSpiral col (t01_a02): {spiral_col}')
print(f't11 cols: {t11_cols}')

In [ ]:
# === STEP 5: KTF3-R SPIN ALIGNMENT TEST ===
results = {}

# Test A: Spiral/disk fraction near vs far
if spiral_col:
    sf = gz2[spiral_col].values
    ok = np.isfinite(sf)
    sn = sf[near & ok]
    sf_ = sf[far  & ok]
    t, p = stats.ttest_ind(sn, sf_)
    print(f'TEST A — Spiral fraction (disk/features) near vs far axis:')
    print(f'  Near: N={len(sn):,}, mean={sn.mean():.5f}')
    print(f'  Far:  N={len(sf_):,}, mean={sf_.mean():.5f}')
    print(f'  Difference: {sn.mean()-sf_.mean():+.5f}')
    print(f'  sigma = {t:.3f},  p = {p:.4e}')
    verdict = '✅ SIGNAL' if abs(t)>2 else ('⚠️ MARGINAL' if abs(t)>1 else '❌ NULL')
    print(f'  Verdict: {verdict}')
    results['A_spiral'] = t
    print()

# Test B: t11 winding asymmetry
if len(t11_cols) >= 2:
    w0 = gz2[t11_cols[0]].values
    w1 = gz2[t11_cols[-1]].values
    ws = w0 - w1
    ok = np.isfinite(ws)
    wn = ws[near & ok]
    wf = ws[far  & ok]
    if len(wn)>5 and len(wf)>5:
        t2, p2 = stats.ttest_ind(wn, wf)
        print(f'TEST B — Arm winding asymmetry near vs far:')
        print(f'  Col1: {t11_cols[0]}')
        print(f'  Col2: {t11_cols[-1]}')
        print(f'  Near: N={len(wn):,}, mean={wn.mean():.5f}')
        print(f'  Far:  N={len(wf):,}, mean={wf.mean():.5f}')
        print(f'  sigma = {t2:.3f},  p = {p2:.4e}')
        verdict2 = '✅ SIGNAL' if abs(t2)>2 else ('⚠️ MARGINAL' if abs(t2)>1 else '❌ NULL')
        print(f'  Verdict: {verdict2}')
        results['B_winding'] = t2
        print()

# Test C: ALL fraction columns — scan for any asymmetry
print('TEST C — Full scan of all fraction columns:')
scan_results = []
for c in fraction_cols:
    v = gz2[c].values
    ok = np.isfinite(v)
    vn = v[near & ok]
    vf = v[far  & ok]
    if len(vn)>10 and len(vf)>10 and vn.std()>0 and vf.std()>0:
        t_s, p_s = stats.ttest_ind(vn, vf)
        scan_results.append((abs(t_s), t_s, c))

scan_results.sort(reverse=True)
print(f'Top 10 columns by |sigma|:')
for _, t_s, c in scan_results[:10]:
    mark = '🔴' if abs(t_s)>3 else ('🟡' if abs(t_s)>2 else '⚪')
    print(f'  {mark} sigma={t_s:+.3f}  {c}')

In [ ]:
# === STEP 6: PLOT ===
bins = np.arange(0, 181, 15)
bc   = 0.5*(bins[:-1]+bins[1:])

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle(f'v64b — KTF³-R Spin Alignment Test (l=277°, b=-31°)', fontweight='bold')

if spiral_col:
    sf = gz2[spiral_col].values
    means, errs, ns = [], [], []
    for i in range(len(bins)-1):
        m = (sep>=bins[i]) & (sep<bins[i+1]) & np.isfinite(sf)
        if m.sum()>10:
            means.append(sf[m].mean())
            errs.append(sf[m].std()/np.sqrt(m.sum()))
            ns.append(m.sum())
        else:
            means.append(np.nan); errs.append(np.nan); ns.append(0)

    ax = axes[0]
    ax.errorbar(bc, means, yerr=errs, fmt='bo-', lw=2, capsize=4, markersize=7)
    ax.axhline(np.nanmean(means), color='gray', ls='--', lw=1.5, label='Mean')
    ax.axvline(NEAR, color='red',    ls=':', lw=2.5, label=f'Near <{NEAR}°')
    ax.axvline(FAR,  color='orange', ls=':', lw=2.5, label=f'Far >{FAR}°')
    sigma_A = results.get('A_spiral', 0)
    ax.set_title(f'Spiral fraction profile  |  σ={sigma_A:.2f}', fontsize=12)
    ax.set_xlabel('Angular separation from KTF³ axis [°]')
    ax.set_ylabel('Mean spiral/disk fraction')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
counts = [((sep>=bins[i])&(sep<bins[i+1])).sum() for i in range(len(bins)-1)]
ax.bar(bc, counts, width=12, color='steelblue', alpha=0.7, edgecolor='navy')
ax.axvline(NEAR, color='red',    ls=':', lw=2.5)
ax.axvline(FAR,  color='orange', ls=':', lw=2.5)
ax.set_xlabel('Angular separation from KTF³ axis [°]')
ax.set_ylabel('Number of galaxies')
ax.set_title(f'Galaxy distribution  |  Near={near.sum():,}, Far={far.sum():,}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v64b_spin_alignment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v64b_spin_alignment.png')

In [ ]:
print('='*60)
print('v64b SUMMARY — KTF³-R Galaxy Spin Alignment')
print('='*60)
print()
print('FRAMEWORK: KTF³-R')
print('  φ_R = rotation by θ_R=25.7° around l=277°, b=-31°')
print('  N=14 cycles → 360° → identity')
print('  Motivated by: v58 NULL (Radojičić 2026, 40B pairs)')
print()
print('DATA: Galaxy Zoo 2 — Hart et al. (2016)')
print(f'  {len(gz2):,} galaxies')
print(f'  Near axis (<{NEAR}°): {near.sum():,}')
print(f'  Far  axis (>{FAR}°):  {far.sum():,}')
print()
print('RESULTS:')
for name, sigma in results.items():
    s = '✅ SIGNAL' if abs(sigma)>2 else ('⚠️ MARGINAL' if abs(sigma)>1 else '❌ NULL')
    print(f'  {name}: σ = {sigma:+.3f}  {s}')
print()
if scan_results:
    best = scan_results[0]
    print(f'Strongest column: σ={best[1]:+.3f}  {best[2]}')
print()
print('NOTE: Hart16 has no direct CW/CCW column.')
print('Direct spin test possible with Galaxy Zoo 1 (CW/CCW explicit)')
print('or Euclid DR1 weak lensing shear (October 2026).')
print()
print('OSF: osf.io/42nks (timestamped 1 April 2026)')
print('GitHub: github.com/Andrew-Cot/KTF3-Analyse')
print('Ref: Radojičić (2026), v58, NULL mirror correlation, 40B pairs')